# Research 33 — VIP × region fourth CV seed

**Hypothesis:** a fourth fixed fold partition may further stabilize the VIP × region model average. Fit the exact balanced-weight LightGBM configuration with CV seed 31415; all features and hyperparameters match notebooks 28, 29, and 31. Save aligned OOF probabilities for one predeclared averaging comparison. Test is not loaded.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)


def make_features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (num, den) in ratios.items():
        x[name] = x[num] / (x[den].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    vip = x["vip_tier"].map(lambda value: "__MISSING__" if pd.isna(value) else str(int(value)))
    region = x["region"].fillna("__MISSING__").astype(str)
    x["vip_region"] = vip + "__" + region
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = make_features(train)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=31415)
folds = list(cv.split(X, y))
p_candidate = np.zeros((len(y), 4), dtype=float)
started = perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
    x_fit, x_valid = X.iloc[fit_idx].copy(), X.iloc[valid_idx].copy()
    for col in cat_cols:
        dtype = pd.CategoricalDtype(categories=sorted(x_fit[col].dropna().unique().tolist()))
        x_fit[col] = x_fit[col].astype(dtype)
        x_valid[col] = x_valid[col].astype(dtype)
    model = LGBMClassifier(
        objective="multiclass",
        class_weight="balanced",
        n_estimators=440,
        learning_rate=0.063863707925569,
        num_leaves=59,
        max_depth=4,
        min_child_samples=81,
        reg_lambda=0.12942956115511214,
        colsample_bytree=0.7975990992289793,
        random_state=42,
        n_jobs=2,
        verbosity=-1,
    )
    model.fit(x_fit, y[fit_idx], categorical_feature=cat_cols)
    p_candidate[valid_idx] = model.predict_proba(x_valid)
    print(
        f"Fold {fold}: macro F1={f1_score(y[valid_idx], p_candidate[valid_idx].argmax(1), average='macro'):.6f}",
        flush=True,
    )

pred = p_candidate.argmax(axis=1)
row = {
    "candidate": "VIP × region seed 31415",
    "macro_f1": f1_score(y, pred, average="macro"),
    "weighted_f1": f1_score(y, pred, average="weighted"),
}
for fold, (_, idx) in enumerate(folds, start=1):
    row[f"fold_{fold}_macro_f1"] = f1_score(y[idx], pred[idx], average="macro")
display(pd.DataFrame([row]).round(6))
pd.DataFrame(p_candidate, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-vip-region-seed-31415-oof.csv", index=False
)
print(f"Elapsed seconds={perf_counter() - started:.1f}. Train OOF only; Test not loaded.")

Fold 1: macro F1=0.741469


Fold 2: macro F1=0.740737


Fold 3: macro F1=0.745158


,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1
0,VIP × region seed 31415,0.742458,0.75443,0.741469,0.740737,0.745158


Elapsed seconds=82.5. Train OOF only; Test not loaded.
